# Logistic Regression and Linear SVM

### Load the Processed Tables
- The splits, the normalised tags and the shared vocabulary all come from the preparation notebook
- Both reference scores are loaded here, because they are what the two linear models have to beat

In [1]:
from pathlib import Path

import pandas as pd

seed = 42

candidate_paths = [
    Path("data/processed"),
    Path("../data/processed"),
]
processed_dir = next((path for path in candidate_paths if path.exists()), None)
if processed_dir is None:
    raise FileNotFoundError(
        "Could not find data/processed. Run data_preparation.ipynb first."
    )
for required in ["nb_bow_summary.csv", "tfidf_vs_bow.csv"]:
    if not (processed_dir / required).exists():
        raise FileNotFoundError(
            "Could not find " + required + ". Run the earlier notebooks first."
        )
splits = pd.read_csv(processed_dir / "splits.csv", dtype={"id": str})
tags_frame = pd.read_csv(processed_dir / "tags_normalised.csv", dtype={"id": str})
vocabulary_frame = pd.read_csv(processed_dir / "shared_vocabulary_50.csv")
bow_reference = pd.read_csv(processed_dir / "nb_bow_summary.csv")
tfidf_reference = pd.read_csv(processed_dir / "tfidf_vs_bow.csv")

tag_vocabulary = list(vocabulary_frame["tag"])

print("Splits shape:", splits.shape)
print("Tags in the shared vocabulary:", len(tag_vocabulary))

Splits shape: (82810, 4)
Tags in the shared vocabulary: 50


### The Two References
- Reference one is bag of words counts with Naive Bayes, which is the first indicator
- Reference two is TF-IDF with Naive Bayes at its jointly selected configuration and cut, which is the second
- Both were scored on validation with the same tags and the same splits, so the numbers are directly comparable

In [2]:
references = []
for record in bow_reference.itertuples(index=False):
    references.append({
        "split": record.split,
        "reference": "naive bayes on counts",
        "micro_f1": record.micro_f1,
        "macro_f1": record.macro_f1,
    })
for record in tfidf_reference.itertuples(index=False):
    references.append({
        "split": record.split,
        "reference": "naive bayes on tfidf",
        "micro_f1": record.tfidf_micro_f1_tuned_cut,
        "macro_f1": record.tfidf_macro_f1_tuned_cut,
    })
reference_table = pd.DataFrame(references)
best_reference = reference_table.loc[reference_table.groupby("split")["micro_f1"].idxmax()]

print("Winning TF-IDF settings carried over from the sweep:")
for record in tfidf_reference.itertuples(index=False):
    print(" ", record.split,
          "ngram", record.best_ngram_range,
          "min_df", record.best_min_df,
          "max_df", record.best_max_df,
          "threshold", record.best_threshold)
print()
print("Best reference micro F1 per split:",
      dict(zip(best_reference["split"], best_reference["micro_f1"])))
reference_table

Winning TF-IDF settings carried over from the sweep:
  random ngram (1, 2) min_df 20 max_df 0.5 threshold 0.05
  artist ngram (1, 2) min_df 20 max_df 0.5 threshold 0.05

Best reference micro F1 per split: {'artist': 0.267, 'random': 0.273}


,split,reference,micro_f1,macro_f1
0,random,naive bayes on counts,0.260,0.152
1,artist,naive bayes on counts,0.239,0.131
2,random,naive bayes on tfidf,0.273,0.084
3,artist,naive bayes on tfidf,0.267,0.079


### Read the Lyrics Back
- The same two cleaning functions as the preparation notebook, so the tokens match
- The sha1 fingerprints and the word counts are checked against lyrics_clean.csv, which confirms each document sits on the row its id belongs to

In [3]:
import hashlib
import re

candidate_paths = [
    Path("music4all-en"),
    Path("../music4all-en"),
]
data_dir = next((path for path in candidate_paths if path.exists()), None)
if data_dir is None:
    raise FileNotFoundError(
        "Could not find music4all-en. Unzip the subset to the repo root."
    )
lyrics_dir = data_dir / "lyrics"
marker_pattern = re.compile(r"^\s*\[[^\]]{1,30}\]\s*$")


def clean_lines(text):
    kept = []
    for line in text.replace("\r\n", "\n").replace("\r", "\n").split("\n"):
        line = line.rstrip()
        if not line.strip():
            continue
        if marker_pattern.match(line):
            continue
        kept.append(line)

    return "\n".join(kept)


def flatten(text):
    return re.sub(r"\s+", " ", clean_lines(text).lower()).strip()


track_ids = list(splits["id"])
documents = []
for track_id in track_ids:
    path = lyrics_dir / (track_id + ".txt")
    raw = path.read_text(encoding="utf-8", errors="replace")
    documents.append(flatten(raw))

word_counts = pd.Series([len(text.split()) for text in documents], index=track_ids)
fingerprints = pd.Series(
    [hashlib.sha1(text.encode("utf-8")).hexdigest() for text in documents],
    index=track_ids,
)
expected = pd.read_csv(processed_dir / "lyrics_clean.csv", dtype={"id": str}).set_index("id")

print("Documents read:", len(documents))
print("Fingerprints matching lyrics_clean.csv:",
      int((fingerprints == expected.loc[track_ids, "fingerprint"]).sum()), "of", len(track_ids))
print("Word counts matching lyrics_clean.csv:",
      int((word_counts == expected.loc[track_ids, "n_words"]).sum()), "of", len(track_ids))

Documents read: 82810
Fingerprints matching lyrics_clean.csv: 82810 of 82810
Word counts matching lyrics_clean.csv: 82810 of 82810


### Build the Target Matrix
- One binary column per tag in the shared vocabulary, one row per track, in the order of splits.csv
- Identical to the two earlier notebooks, so every score in this one lines up with theirs

In [4]:
import numpy as np

tag_lists = {}
for record in tags_frame.itertuples(index=False):
    if isinstance(record.tags, str) and record.tags != "":
        tag_lists[record.id] = record.tags.split(",")
    else:
        tag_lists[record.id] = []

position = {}
for i in range(len(tag_vocabulary)):
    position[tag_vocabulary[i]] = i

target = np.zeros((len(track_ids), len(tag_vocabulary)), dtype=np.int8)
row = 0
for track_id in track_ids:
    for tag in tag_lists[track_id]:
        if tag in position:
            target[row, position[tag]] = 1
    row = row + 1

print("Target shape:", target.shape)
print("Cardinality:", round(float(target.sum(axis=1).mean()), 2))

Target shape: (82810, 50)
Cardinality: 2.46


### Index the Two Splits
- Row positions rather than ids, so the feature matrix and the target can be sliced the same way
- The test rows are counted here but never fed to a model, because everything is selected on validation

In [5]:
split_rows = {}
for name in ["random", "artist"]:
    column = splits[name + "_split"]
    parts = {}
    for part in ["train", "valid", "test"]:
        parts[part] = np.flatnonzero(column == part)
    split_rows[name] = parts

    print(name, "split:",
          "train", len(parts["train"]),
          "valid", len(parts["valid"]),
          "test", len(parts["test"]))

random split: train 57966 valid 12421 test 12423
artist split: train 57997 valid 12424 test 12389


### The Two Feature Sets
- Counts are the features reference one used, with every vectoriser parameter left at its default
- TF-IDF is the configuration the sweep selected, so each linear model is measured against each reference on matching features
- Holding the features fixed is what makes the comparison about the classifier rather than about the representation

In [6]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

documents = np.array(documents, dtype=object)
winning = tfidf_reference.set_index("split")

feature_sets = {}
for name in ["random", "artist"]:
    parts = split_rows[name]
    settings = winning.loc[name]
    counts = CountVectorizer()
    tfidf = TfidfVectorizer(
        ngram_range=tuple(int(part) for part in settings["best_ngram_range"].strip("()").split(",")),
        min_df=int(settings["best_min_df"]),
        max_df=float(settings["best_max_df"]),
    )
    built = {}
    for label, vectoriser in [("counts", counts), ("tfidf", tfidf)]:
        X_train = vectoriser.fit_transform(documents[parts["train"]])
        X_valid = vectoriser.transform(documents[parts["valid"]])
        built[label] = (X_train, X_valid)

        print(name, label, "terms", X_train.shape[1],
              "mass per document", round(float(X_train.sum(axis=1).mean()), 2))
    feature_sets[name] = built

random counts terms 86194 mass per document 230.02


random tfidf terms 53828 mass per document 8.89


artist counts terms 85307 mass per document 231.76


artist tfidf terms 54169 mass per document 8.9


### Train the Two Linear Models per Tag
- One classifier per tag again, so the only thing that changes from the references is the learner
- Logistic regression is cut on its predicted probability, the same grid the TF-IDF sweep used
- The SVM has no probability to cut, so its margin is swept instead, with zero being the default decision
- Convergence is recorded rather than assumed, because raw counts are badly scaled for either model

In [7]:
import time
import warnings

from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support
from sklearn.svm import LinearSVC

probability_cuts = [0.5, 0.3, 0.2, 0.15, 0.1, 0.05, 0.02]
margin_cuts = [0.0, -0.1, -0.25, -0.5, -0.75, -1.0]

model_rows = []
convergence_rows = []
for name in ["random", "artist"]:
    parts = split_rows[name]
    y_train = target[parts["train"]]
    y_valid = target[parts["valid"]]
    for features in ["counts", "tfidf"]:
        X_train, X_valid = feature_sets[name][features]
        for learner in ["logistic", "svm"]:
            started = time.perf_counter()
            scores = np.zeros((X_valid.shape[0], len(tag_vocabulary)))
            not_converged = 0
            for j in range(len(tag_vocabulary)):
                if learner == "logistic":
                    model = LogisticRegression(max_iter=2000, random_state=seed)
                else:
                    model = LinearSVC(max_iter=2000, random_state=seed)
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always", ConvergenceWarning)
                    model.fit(X_train, y_train[:, j])
                    for item in caught:
                        if issubclass(item.category, ConvergenceWarning):
                            not_converged = not_converged + 1
                if learner == "logistic":
                    scores[:, j] = model.predict_proba(X_valid)[:, 1]
                else:
                    scores[:, j] = model.decision_function(X_valid)
            elapsed = time.perf_counter() - started
            convergence_rows.append({
                "split": name,
                "features": features,
                "model": learner,
                "tags_not_converged": not_converged,
                "seconds": round(elapsed, 1),
            })
            if learner == "logistic":
                cuts = probability_cuts
            else:
                cuts = margin_cuts
            for cut in cuts:
                predicted = (scores >= cut).astype(np.int8)
                micro = precision_recall_fscore_support(
                    y_valid, predicted, average="micro", zero_division=0,
                )
                macro = precision_recall_fscore_support(
                    y_valid, predicted, average="macro", zero_division=0,
                )
                model_rows.append({
                    "split": name,
                    "features": features,
                    "model": learner,
                    "cut": cut,
                    "default_cut": cut == cuts[0],
                    "micro_precision": round(micro[0], 3),
                    "micro_recall": round(micro[1], 3),
                    "micro_f1": round(micro[2], 3),
                    "macro_f1": round(macro[2], 3),
                    "predicted": int(predicted.sum()),
                })
            best_here = max(
                [r for r in model_rows
                 if r["split"] == name and r["features"] == features and r["model"] == learner],
                key=lambda r: r["micro_f1"],
            )

            print(name, features, learner,
                  "best micro F1", best_here["micro_f1"],
                  "at cut", best_here["cut"],
                  "tags not converged", not_converged,
                  "seconds", round(elapsed, 1))
linear = pd.DataFrame(model_rows)
convergence = pd.DataFrame(convergence_rows)

print()
print("Scored rows:", len(linear))

random counts logistic best micro F1 0.256 at cut 0.15 tags not converged 1 seconds 1031.4


random counts svm best micro F1 0.216 at cut -0.5 tags not converged 50 seconds 1124.9


random tfidf logistic best micro F1 0.32 at cut 0.15 tags not converged 0 seconds 32.6


random tfidf svm best micro F1 0.285 at cut -0.5 tags not converged 0 seconds 89.3


artist counts logistic best micro F1 0.235 at cut 0.15 tags not converged 3 seconds 876.5


artist counts svm best micro F1 0.198 at cut -0.75 tags not converged 50 seconds 754.5


artist tfidf logistic best micro F1 0.309 at cut 0.15 tags not converged 0 seconds 33.6


artist tfidf svm best micro F1 0.262 at cut -0.5 tags not converged 0 seconds 89.5

Scored rows: 52


### Convergence
- Raw counts carry a few hundred units per document and TF-IDF rows are normalised, so the two feature sets are on very different scales
- A tag that did not converge within the iteration cap still produced a model, so its score is kept but should be read with that in mind

In [8]:
print(convergence.to_string(index=False))

print()
print("Tags that failed to converge, by feature set:")
print(convergence.groupby("features")["tags_not_converged"].sum().to_string())

 split features    model  tags_not_converged  seconds
random   counts logistic                   1   1031.4
random   counts      svm                  50   1124.9
random    tfidf logistic                   0     32.6
random    tfidf      svm                   0     89.3
artist   counts logistic                   3    876.5
artist   counts      svm                  50    754.5
artist    tfidf logistic                   0     33.6
artist    tfidf      svm                   0     89.5

Tags that failed to converge, by feature set:
features
counts    104
tfidf       0


### What the Default Cut Would Have Concluded
- The default is 0.5 for logistic regression and a margin of zero for the SVM
- The earlier sweep showed that a default cut flatters whichever model happens to suit it, so both cuts are reported

In [9]:
default_only = linear[linear["default_cut"]].copy()
tuned = linear.loc[linear.groupby(["split", "features", "model"])["micro_f1"].idxmax()].copy()

comparison = default_only.merge(
    tuned[["split", "features", "model", "cut", "micro_f1", "macro_f1"]],
    on=["split", "features", "model"],
    suffixes=("_default", "_tuned"),
)
columns = ["split", "features", "model", "micro_f1_default", "micro_f1_tuned",
           "macro_f1_default", "macro_f1_tuned", "cut_tuned"]
print(comparison[columns].to_string(index=False))

 split features    model  micro_f1_default  micro_f1_tuned  macro_f1_default  macro_f1_tuned  cut_tuned
random   counts logistic             0.200           0.256             0.151           0.192       0.15
random   counts      svm             0.198           0.216             0.160           0.171      -0.50
random    tfidf logistic             0.067           0.320             0.037           0.169       0.15
random    tfidf      svm             0.171           0.285             0.104           0.209      -0.50
artist   counts logistic             0.174           0.235             0.128           0.167       0.15
artist   counts      svm             0.173           0.198             0.132           0.149      -0.75
artist    tfidf logistic             0.063           0.309             0.037           0.149       0.15
artist    tfidf      svm             0.147           0.262             0.078           0.176      -0.50


### Compare Against Both References
- Each linear model is placed next to the reference that shares its features, which is the like for like comparison
- A positive gain means the linear model beat the Naive Bayes reference on the same representation

In [10]:
reference_lookup = {}
for record in reference_table.itertuples(index=False):
    if record.reference == "naive bayes on counts":
        reference_lookup[(record.split, "counts")] = (record.micro_f1, record.macro_f1)
    else:
        reference_lookup[(record.split, "tfidf")] = (record.micro_f1, record.macro_f1)

against_rows = []
for record in tuned.itertuples(index=False):
    reference_micro, reference_macro = reference_lookup[(record.split, record.features)]
    against_rows.append({
        "split": record.split,
        "features": record.features,
        "model": record.model,
        "cut": record.cut,
        "micro_f1": record.micro_f1,
        "reference_micro_f1": reference_micro,
        "micro_gain": round(record.micro_f1 - reference_micro, 3),
        "macro_f1": record.macro_f1,
        "reference_macro_f1": reference_macro,
        "macro_gain": round(record.macro_f1 - reference_macro, 3),
    })
against = pd.DataFrame(against_rows).sort_values(["split", "features", "model"])

print(against.to_string(index=False))

print()
print("Beat their reference on micro F1:", int((against["micro_gain"] > 0).sum()), "of", len(against))
print("Beat their reference on macro F1:", int((against["macro_gain"] > 0).sum()), "of", len(against))

 split features    model   cut  micro_f1  reference_micro_f1  micro_gain  macro_f1  reference_macro_f1  macro_gain
artist   counts logistic  0.15     0.235               0.239      -0.004     0.167               0.131       0.036
artist   counts      svm -0.75     0.198               0.239      -0.041     0.149               0.131       0.018
artist    tfidf logistic  0.15     0.309               0.267       0.042     0.149               0.079       0.070
artist    tfidf      svm -0.50     0.262               0.267      -0.005     0.176               0.079       0.097
random   counts logistic  0.15     0.256               0.260      -0.004     0.192               0.152       0.040
random   counts      svm -0.50     0.216               0.260      -0.044     0.171               0.152       0.019
random    tfidf logistic  0.15     0.320               0.273       0.047     0.169               0.084       0.085
random    tfidf      svm -0.50     0.285               0.273       0.012     0.2

### The Best Model Overall
- Taken across both feature sets and both learners, per split, on validation micro F1
- The reference row beside it is the stronger of the two Naive Bayes points, so this is the honest headline comparison

In [11]:
overall_rows = []
for name in ["random", "artist"]:
    here = tuned[tuned["split"] == name]
    winner = here.loc[here["micro_f1"].idxmax()]
    reference = best_reference[best_reference["split"] == name].iloc[0]
    overall_rows.append({
        "split": name,
        "best_model": winner["model"],
        "best_features": winner["features"],
        "best_cut": winner["cut"],
        "best_micro_f1": winner["micro_f1"],
        "best_macro_f1": winner["macro_f1"],
        "best_reference": reference["reference"],
        "reference_micro_f1": reference["micro_f1"],
        "reference_macro_f1": reference["macro_f1"],
        "micro_gain": round(winner["micro_f1"] - reference["micro_f1"], 3),
        "macro_gain": round(winner["macro_f1"] - reference["macro_f1"], 3),
    })
overall = pd.DataFrame(overall_rows)

print(overall.to_string(index=False))

 split best_model best_features  best_cut  best_micro_f1  best_macro_f1       best_reference  reference_micro_f1  reference_macro_f1  micro_gain  macro_gain
random   logistic         tfidf      0.15          0.320          0.169 naive bayes on tfidf               0.273               0.084       0.047       0.085
artist   logistic         tfidf      0.15          0.309          0.149 naive bayes on tfidf               0.267               0.079       0.042       0.070


### Per Tag Scores for the Winning Model
- The winning combination is refitted so the per tag table has the same shape as the ones the earlier notebooks saved

In [12]:
tag_tables = {}
for name in ["random", "artist"]:
    parts = split_rows[name]
    y_train = target[parts["train"]]
    y_valid = target[parts["valid"]]
    winner = overall[overall["split"] == name].iloc[0]
    X_train, X_valid = feature_sets[name][winner["best_features"]]
    predicted = np.zeros((X_valid.shape[0], len(tag_vocabulary)), dtype=np.int8)
    for j in range(len(tag_vocabulary)):
        if winner["best_model"] == "logistic":
            model = LogisticRegression(max_iter=2000, random_state=seed)
        else:
            model = LinearSVC(max_iter=2000, random_state=seed)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            model.fit(X_train, y_train[:, j])
        if winner["best_model"] == "logistic":
            column = model.predict_proba(X_valid)[:, 1]
        else:
            column = model.decision_function(X_valid)
        predicted[:, j] = (column >= winner["best_cut"]).astype(np.int8)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_valid, predicted, average=None, zero_division=0,
    )
    tag_tables[name] = pd.DataFrame({
        "tag": tag_vocabulary,
        "precision": precision.round(3),
        "recall": recall.round(3),
        "f1": f1.round(3),
        "support": support,
    }).sort_values("f1", ascending=False).reset_index(drop=True)

    print(name, "best five tags:", list(tag_tables[name]["tag"].head(5)))
    print(name, "worst five tags:", list(tag_tables[name]["tag"].tail(5)))
tag_tables["random"].head(10)

random best five tags: ['rap', 'hip-hop', 'pop', 'rock', 'death metal']
random worst five tags: ['psychedelic', 'lo-fi', 'pop rock', 'fip', 'shoegaze']


artist best five tags: ['rap', 'hip-hop', 'rock', 'pop', 'death metal']
artist worst five tags: ['psychedelic', 'lo-fi', 'pop rock', 'fip', 'shoegaze']


,tag,precision,recall,f1,support
0,rap,0.684,0.666,0.675,455
1,hip-hop,0.701,0.650,0.674,594
2,pop,0.314,0.817,0.453,2362
3,rock,0.295,0.911,0.445,3161
4,death metal,0.408,0.473,0.438,273
5,metal,0.348,0.529,0.420,852
6,indie,0.266,0.773,0.396,2159
7,classic rock,0.318,0.300,0.309,733
8,female vocalists,0.218,0.435,0.290,1394
9,dance,0.294,0.271,0.282,532


### Where the Linear Models Help and Where They Do Not
- Tag by tag against both references, so it is clear whether a gain is spread out or concentrated in a few tags
- The per tag tables of both references were saved by the earlier notebooks, so nothing is refitted here
- Tags where both models score zero are counted as level rather than improved, because neither one predicts them at all

In [13]:
change_rows = []
for name in ["random", "artist"]:
    bow_tags = pd.read_csv(processed_dir / ("nb_bow_per_tag_" + name + ".csv"))
    tfidf_tags = pd.read_csv(processed_dir / ("tfidf_per_tag_" + name + ".csv"))
    merged = tag_tables[name][["tag", "f1", "support"]].rename(columns={"f1": "f1_linear"})
    merged = merged.merge(bow_tags[["tag", "f1"]].rename(columns={"f1": "f1_bow"}), on="tag")
    merged = merged.merge(tfidf_tags[["tag", "f1"]].rename(columns={"f1": "f1_tfidf"}), on="tag")
    merged["change_vs_bow"] = (merged["f1_linear"] - merged["f1_bow"]).round(3)
    merged["change_vs_tfidf"] = (merged["f1_linear"] - merged["f1_tfidf"]).round(3)
    merged.insert(0, "split", name)
    change_rows.append(merged)

    for column, label in [("change_vs_bow", "counts"), ("change_vs_tfidf", "tfidf")]:
        print(name, "against naive bayes on", label + ":",
              "improved", int((merged[column] > 0).sum()),
              "worsened", int((merged[column] < 0).sum()),
              "level", int((merged[column] == 0).sum()),
              "mean change", round(float(merged[column].mean()), 3))
    print(name, "tags scored zero:",
          "winning model", int((merged["f1_linear"] == 0).sum()),
          "counts reference", int((merged["f1_bow"] == 0).sum()),
          "tfidf reference", int((merged["f1_tfidf"] == 0).sum()))
per_tag_change = pd.concat(change_rows, ignore_index=True)
biggest = per_tag_change.reindex(
    per_tag_change["change_vs_bow"].abs().sort_values(ascending=False).index
)

print()
print("Largest movements against the bag of words reference:")
print(biggest.head(10).to_string(index=False))

random against naive bayes on counts: improved 28 worsened 22 level 0 mean change 0.017
random against naive bayes on tfidf: improved 41 worsened 0 level 9 mean change 0.085
random tags scored zero: winning model 9 counts reference 0 tfidf reference 23
artist against naive bayes on counts: improved 29 worsened 19 level 2 mean change 0.018
artist against naive bayes on tfidf: improved 39 worsened 0 level 11 mean change 0.07
artist tags scored zero: winning model 11 counts reference 2 tfidf reference 24

Largest movements against the bag of words reference:
 split           tag  f1_linear  support  f1_bow  f1_tfidf  change_vs_bow  change_vs_tfidf
artist           rap      0.661      358   0.502     0.562          0.159            0.099
artist       hip-hop      0.623      458   0.480     0.516          0.143            0.107
random post-hardcore      0.210      372   0.079     0.000          0.131            0.210
random           rap      0.675      455   0.557     0.586          0.118 

### Save
- The full grid, the convergence record, the comparison against both references and the per tag tables

In [14]:
linear.to_csv(processed_dir / "linear_models.csv", index=False)
convergence.to_csv(processed_dir / "linear_convergence.csv", index=False)
against.to_csv(processed_dir / "linear_vs_references.csv", index=False)
overall.to_csv(processed_dir / "linear_best_overall.csv", index=False)
per_tag_change.to_csv(processed_dir / "linear_per_tag_change.csv", index=False)
for name in ["random", "artist"]:
    tag_tables[name].to_csv(processed_dir / ("linear_per_tag_" + name + ".csv"), index=False)

print("Wrote: linear_models.csv")
print("Wrote: linear_convergence.csv")
print("Wrote: linear_vs_references.csv")
print("Wrote: linear_best_overall.csv")
print("Wrote: linear_per_tag_change.csv")
print("Wrote: linear_per_tag_random.csv")
print("Wrote: linear_per_tag_artist.csv")

Wrote: linear_models.csv
Wrote: linear_convergence.csv
Wrote: linear_vs_references.csv
Wrote: linear_best_overall.csv
Wrote: linear_per_tag_change.csv
Wrote: linear_per_tag_random.csv
Wrote: linear_per_tag_artist.csv
